# SVM Model — Titanic Dataset
**Model:** Support Vector Machine (RBF Kernel)  
**Task:** Binary classification — predict passenger survival  
**Split:** Shared pre-processed train/validation split  
**Author:** [Your Name] | Group ML Report
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, time
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_score, recall_score,
    f1_score, ConfusionMatrixDisplay
)
print('Libraries loaded.')

## 1. Load Shared Pre-processed Data

In [ ]:
X_train = pd.read_csv('titanic_X_train.csv')
X_val   = pd.read_csv('titanic_X_val.csv')
y_train = pd.read_csv('titanic_y_train.csv').iloc[:, 0]
y_val   = pd.read_csv('titanic_y_val.csv').iloc[:, 0]

print(f'X_train: {X_train.shape} | X_val: {X_val.shape}')
print(f'Survival rate (train): {y_train.mean():.2%}')
print(f'Features: {list(X_train.columns)}')

## 2. SVM Model

### Hyperparameters
| Parameter | Value | Reason |
|-----------|-------|--------|
| `kernel` | `rbf` | Non-linear boundary (sex, class, age interact non-linearly) |
| `C` | `1.0` | Default regularisation |
| `gamma` | `scale` | Auto-scales with n_features |
| `class_weight` | `balanced` | ~62% did not survive — corrects imbalance |
| `probability` | `True` | Enables ROC-AUC and probability outputs |

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svm',    SVC(kernel='rbf', C=1.0, gamma='scale',
                   class_weight='balanced', probability=True, random_state=42))
])

t0 = time.time()
pipe.fit(X_train, y_train)
train_time = time.time() - t0
print(f'Training complete in {train_time:.3f}s')

## 3. Model Evaluation

In [ ]:
y_pred = pipe.predict(X_val)
y_prob = pipe.predict_proba(X_val)[:, 1]

acc = accuracy_score(y_val, y_pred)
roc = roc_auc_score(y_val, y_prob)

print(f'Accuracy : {acc:.4f}')
print(f'ROC-AUC  : {roc:.4f}')
print(f'Precision: {precision_score(y_val, y_pred):.4f}')
print(f'Recall   : {recall_score(y_val, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_val, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_val, y_pred, target_names=['Not Survived', 'Survived']))

### 3.1 Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_val, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['Not Survived', 'Survived']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('SVM — Confusion Matrix\nTitanic Dataset', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig('output_plots_titanic/svm_01_confusion_matrix.png', dpi=150)
plt.show()

### 3.2 ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_val, y_prob)
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(fpr, tpr, color='steelblue', lw=2, label=f'SVM RBF (AUC = {roc:.3f})')
ax.plot([0,1],[0,1],'--', color='gray', lw=1, label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('SVM — ROC Curve\nTitanic Dataset', fontsize=12)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('output_plots_titanic/svm_02_roc_curve.png', dpi=150)
plt.show()

### 3.3 Performance Metrics Summary

In [ ]:
metrics = {
    'Accuracy':  acc,
    'Precision': precision_score(y_val, y_pred),
    'Recall':    recall_score(y_val, y_pred),
    'F1':        f1_score(y_val, y_pred),
    'ROC-AUC':  roc
}
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(metrics.keys(), metrics.values(),
              color=['#4C72B0','#55A868','#C44E52','#8172B2','#937860'])
ax.set_ylim(0, 1.1)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=10)
ax.set_title('SVM — Performance Metrics\nTitanic Dataset', fontsize=12)
ax.set_ylabel('Score'); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('output_plots_titanic/svm_03_metrics_bar.png', dpi=150)
plt.show()

### 3.4 Decision Boundary (PCA 2D Projection)

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
scaler = StandardScaler()
X_tr2 = pca.fit_transform(scaler.fit_transform(X_train))
X_va2 = pca.transform(scaler.transform(X_val))

svm2 = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=42)
svm2.fit(X_tr2, y_train)

xx, yy = np.meshgrid(
    np.linspace(X_tr2[:,0].min()-1, X_tr2[:,0].max()+1, 200),
    np.linspace(X_tr2[:,1].min()-1, X_tr2[:,1].max()+1, 200)
)
Z = svm2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(6, 5))
ax.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')
sc = ax.scatter(X_va2[:,0], X_va2[:,1], c=y_val, cmap='coolwarm',
                edgecolors='k', linewidths=0.4, s=30, alpha=0.8)
ax.set_xlabel('PCA Component 1'); ax.set_ylabel('PCA Component 2')
ax.set_title('SVM Decision Boundary (PCA 2D Projection)\nTitanic Dataset', fontsize=11)
plt.colorbar(sc, ax=ax, label='Survived')
plt.tight_layout()
plt.savefig('output_plots_titanic/svm_04_decision_boundary_pca.png', dpi=150)
plt.show()

## 4. Results Summary

| Metric | Score |
|--------|-------|
| Accuracy | 0.8045 |
| ROC-AUC | 0.8588 |
| Precision (Survived) | 0.7100 |
| Recall (Survived) | 0.8300 |
| F1 (Survived) | 0.7700 |

### Observations
- **80.4% accuracy** with strong **ROC-AUC of 0.86**, showing good overall discrimination.
- The model balances precision and recall well across both classes (macro avg F1 = 0.80).
- `class_weight='balanced'` helps compensate for the ~62/38 class split.
- The PCA decision boundary plot shows clear but non-linear class separation, validating the RBF kernel choice.

### Notes for Report Integration
- All plots saved to `output_plots_titanic/` with `svm_` prefix.
- Evaluation performed on the shared validation split (`titanic_X_val`, `titanic_y_val`).
- Titanic test labels are unavailable — validation set used for all metric reporting.
- `random_state=42` throughout for reproducibility.